### Matrix summation with a 2D grid and 2D blocks. Adapt it to integer matrix addition. Find the best execution configuration.

### Name: Chidroop M J
### Registered number: 212225240029

In [5]:
%%writefile sum_matrix.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/time.h>

#define CHECK(call)                                                            \
{                                                                              \
    const cudaError_t error = call;                                            \
    if (error != cudaSuccess)                                                  \
    {                                                                          \
        fprintf(stderr, "Error: %s:%d, ", __FILE__, __LINE__);                 \
        fprintf(stderr, "code: %d, reason: %s\n", error,                       \
                cudaGetErrorString(error));                                    \
        exit(1);                                                               \
    }                                                                          \
}

inline double seconds()
{
    struct timeval tp;
    struct timezone tzp;
    gettimeofday(&tp, &tzp);
    return ((double)tp.tv_sec + (double)tp.tv_usec * 1.e-6);
}

void initialData(int *ip, const int size)
{
    for (int i = 0; i < size; i++)
        ip[i] = (int)(rand() & 0xFF) / 10.0f;
}

void sumMatrixOnHost(int *A, int *B, int *C, const int nx, const int ny)
{
    int *ia = A, *ib = B, *ic = C;
    for (int iy = 0; iy < ny; iy++) {
        for (int ix = 0; ix < nx; ix++)
            ic[ix] = ia[ix] + ib[ix];
        ia += nx; ib += nx; ic += nx;
    }
}

void checkResult(int *hostRef, int *gpuRef, const int N)
{
    double epsilon = 1.0E-8;
    bool match = true;
    for (int i = 0; i < N; i++) {
        if (abs(hostRef[i] - gpuRef[i]) > epsilon) {
            match = false;
            printf("host %d gpu %d\n", hostRef[i], gpuRef[i]);
            break;
        }
    }
    printf(match ? "Arrays match.\n\n" : "Arrays do not match.\n\n");
}

__global__ void sumMatrixOnGPU2D(int *A, int *B, int *C, int NX, int NY)
{
    unsigned int ix = blockIdx.x * blockDim.x + threadIdx.x;
    unsigned int iy = blockIdx.y * blockDim.y + threadIdx.y;
    unsigned int idx = iy * NX + ix;
    if (ix < NX && iy < NY)
        C[idx] = A[idx] + B[idx];
}

int main(int argc, char **argv)
{
    printf("%s Starting...\n", argv[0]);

    int dev = 0;
    cudaDeviceProp deviceProp;
    CHECK(cudaGetDeviceProperties(&deviceProp, dev));
    printf("Using Device %d: %s\n", dev, deviceProp.name);
    CHECK(cudaSetDevice(dev));

    /* Reduced size for Colab (T4 has limited host RAM headroom) */
    int nx = 1 << 12;   /* 4096 */
    int ny = 1 << 12;   /* 4096 */
    int nxy = nx * ny;
    int nBytes = nxy * sizeof(int);   /* FIX: was sizeof(float) */

    printf("Matrix size: nx %d ny %d (%.2f MB per array)\n",
           nx, ny, nBytes / 1e6);

    int *h_A, *h_B, *hostRef, *gpuRef;
    h_A     = (int *)malloc(nBytes);
    h_B     = (int *)malloc(nBytes);
    hostRef = (int *)malloc(nBytes);
    gpuRef  = (int *)malloc(nBytes);

    double iStart = seconds();
    initialData(h_A, nxy);
    initialData(h_B, nxy);
    printf("Matrix initialization elapsed %f sec\n", seconds() - iStart);

    memset(hostRef, 0, nBytes);
    memset(gpuRef,  0, nBytes);

    iStart = seconds();
    sumMatrixOnHost(h_A, h_B, hostRef, nx, ny);
    printf("sumMatrixOnHost elapsed %f sec\n", seconds() - iStart);

    int *d_MatA, *d_MatB, *d_MatC;
    CHECK(cudaMalloc((void **)&d_MatA, nBytes));
    CHECK(cudaMalloc((void **)&d_MatB, nBytes));
    CHECK(cudaMalloc((void **)&d_MatC, nBytes));

    CHECK(cudaMemcpy(d_MatA, h_A, nBytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_MatB, h_B, nBytes, cudaMemcpyHostToDevice));

    dim3 block(32, 32);
    dim3 grid((nx + block.x - 1) / block.x, (ny + block.y - 1) / block.y);

    iStart = seconds();
    sumMatrixOnGPU2D<<<grid, block>>>(d_MatA, d_MatB, d_MatC, nx, ny);
    CHECK(cudaDeviceSynchronize());
    printf("sumMatrixOnGPU2D <<<(%d,%d), (%d,%d)>>> elapsed %f sec\n",
           grid.x, grid.y, block.x, block.y, seconds() - iStart);
    CHECK(cudaGetLastError());

    CHECK(cudaMemcpy(gpuRef, d_MatC, nBytes, cudaMemcpyDeviceToHost));

    checkResult(hostRef, gpuRef, nxy);

    CHECK(cudaFree(d_MatA));
    CHECK(cudaFree(d_MatB));
    CHECK(cudaFree(d_MatC));
    free(h_A); free(h_B); free(hostRef); free(gpuRef);

    CHECK(cudaDeviceReset());
    return 0;
}

Overwriting sum_matrix.cu


In [6]:
!nvcc -arch=sm_75 -O2 -o sum_matrix sum_matrix.cu

In [7]:
!./sum_matrix

./sum_matrix Starting...
Using Device 0: Tesla T4
Matrix size: nx 4096 ny 4096 (67.11 MB per array)
Matrix initialization elapsed 0.684771 sec
sumMatrixOnHost elapsed 0.042049 sec
sumMatrixOnGPU2D <<<(128,128), (32,32)>>> elapsed 0.000961 sec
Arrays match.

